<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); color: white; padding: 40px; margin: -10px -10px 20px -10px; border-radius: 0 0 15px 15px;">
<h1 style="margin: 0; font-size: 2.5em;">Testing Superconducting Circuits</h1>
<p style="margin: 10px 0 0 0; font-size: 1.2em; opacity: 0.9;">Week 4 — SCE Futures</p>
</div>

## Contents

- [Learning Objectives](#learning-objectives)

**Part 1: Test Methodology**
1. [Test Environment](#1-test-environment)
2. [DC Characterization](#2-dc-characterization)
3. [Margin Testing](#3-margin-testing)
4. [Functional Testing](#4-functional-testing)
5. [Speed and Timing](#5-speed-timing)
6. [On-Chip Test Structures](#6-test-structures)
7. [Common Failure Modes](#7-failure-modes)
8. [Debug Strategies](#8-debug-strategies)
9. [Yield and Statistics](#9-yield-statistics)

**Part 2: Test Equipment Construction**
10. [LHe Dip Probe Design](#10-lhe-probe)
11. [Rackmount Cryocooler Systems](#11-cryocooler-systems)
12. [Thermal Design](#12-thermal-design)
13. [Signal Integrity](#13-signal-integrity)
14. [Grounding and Shielding](#14-grounding-shielding)
15. [Practical Wiring](#15-practical-wiring)
16. [Magnetic Shielding](#16-magnetic-shielding)
17. [System Integration Checklist](#17-integration-checklist)
18. [Equipment and Vendor Guide](#18-equipment-vendors)
    - Automated SCE Test Systems (OCTOPUX, ICE-T)

---
<a id="learning-objectives"></a>
## Learning Objectives

By the end of this session, you will be able to:

**Part 1: Test Methodology**
- Set up and operate a cryogenic test environment
- Perform DC characterization of Josephson junctions
- Measure and interpret operating margins
- Design effective on-chip test structures
- Diagnose common failure modes in SCE circuits
- Apply systematic debug strategies
- Understand yield metrics and statistical analysis

**Part 2: Test Equipment Construction**
- Design and build an LHe dip probe
- Configure a rackmount cryocooler test system
- Calculate and manage thermal budgets
- Implement proper signal integrity (coax selection, filtering)
- Design star-ground systems and avoid ground loops
- Construct multi-layer magnetic shielding
- Wire a complete cryogenic test system

---

In [ ]:
# Setup
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, Circle, Rectangle
import numpy as np

COLORS = {
    'primary': '#2196F3',
    'secondary': '#FF9800',
    'success': '#4CAF50',
    'danger': '#f44336',
    'warning': '#FFC107',
    'dark': '#1a1a2e',
    'light': '#f5f5f5',
    'aqfp': '#00BCD4',
    'rsfq': '#9C27B0'
}

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.size'] = 11

# Physical constants
Phi_0 = 2.067833848e-15  # Wb
k_B = 1.380649e-23       # J/K

print("Setup complete.")

---
<a id="1-test-environment"></a>
# 1. Test Environment
---

Testing superconducting circuits requires specialized cryogenic infrastructure.

### Cryogenic Systems

| System | Base Temp | Hold Time | Use Case |
|--------|-----------|-----------|----------|
| **LHe Dewar (dip probe)** | 4.2 K | Hours | Quick characterization |
| **Closed-cycle cryocooler** | 3-4 K | Continuous | Production testing |
| **Dilution refrigerator** | 10-20 mK | Continuous | Quantum circuits |
| **ADR (Adiabatic Demag)** | <100 mK | Hours | Low-T characterization |

For AQFP and RSFQ testing, **4.2 K systems** (liquid helium or cryocoolers) are standard.

### Test Setup Block Diagram

```
    Room Temperature                      4.2 K
    ───────────────────────────────────────────────────────
    
    ┌─────────────┐     Coax/Flex      ┌─────────────────┐
    │   Pattern   │─────────────────►  │                 │
    │  Generator  │                    │   Magnetic      │
    └─────────────┘                    │   Shielding     │
                                       │  ┌───────────┐  │
    ┌─────────────┐     Filtered       │  │           │  │
    │    Bias     │─────────────────►  │  │    DUT    │  │
    │   Sources   │     lines          │  │   (Chip)  │  │
    └─────────────┘                    │  │           │  │
                                       │  └───────────┘  │
    ┌─────────────┐                    │        │        │
    │   Readout   │◄─────────────────  │        ▼        │
    │   (Scope/   │     Output         │   Cryo Amps     │
    │   Counter)  │     lines          │   (optional)    │
    └─────────────┘                    └─────────────────┘
```

### Wiring Considerations

| Wire Type | Thermal Load | Bandwidth | Use |
|-----------|--------------|-----------|-----|
| **Stainless coax** | Low | DC-1 GHz | Bias, slow signals |
| **NbTi coax** | Very low | DC-10 GHz | SC below 4K, moderate BW |
| **CuNi coax** | Medium | DC-20 GHz | High-speed I/O |
| **Phosphor bronze twisted pair** | Low | DC-1 MHz | DC bias |

### Thermal Budget

Every wire conducts heat. A 4K cryocooler typically provides:
- **1st stage (40-50 K)**: 30-50 W
- **2nd stage (4 K)**: 1-2 W

Heat loads to manage:
- Wiring: ~1-10 mW per coax
- Radiation: Minimized by shielding
- Chip dissipation: µW to mW (depends on circuit)

---
<a id="2-dc-characterization"></a>
# 2. DC Characterization
---

DC measurements verify that Josephson junctions and basic circuit elements are functional.

### Junction I-V Curve

The most fundamental measurement is the current-voltage characteristic:

```
         I
         ▲
         │         ╱
         │        ╱   ← Normal state (R_n slope)
    I_c ─┼───────●
         │       │
         │       │    ← Supercurrent branch (V=0)
         │       │
    ─────┼───────┼────────► V
         │       │
         │       │
   -I_c ─┼───────●
         │        ╲
         │         ╲
```

**Key parameters extracted:**

| Parameter | Symbol | Typical Value | What It Tells You |
|-----------|--------|---------------|-------------------|
| Critical current | I_c | 50-500 µA | Junction size, quality |
| Normal resistance | R_n | 1-20 Ω | Barrier quality |
| I_c × R_n product | - | 1-2 mV | Material quality figure of merit |
| Subgap resistance | R_sg | >10 × R_n | Junction quality |
| Gap voltage | V_g | 2.8 mV (Nb) | 2Δ/e, superconductor gap |

In [ ]:
# Simulate and plot a JJ I-V curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Ideal I-V curve
I_c = 200e-6  # 200 µA
R_n = 5       # 5 Ω
V_g = 2.8e-3  # 2.8 mV gap voltage

# Supercurrent branch
I_super = np.linspace(-I_c, I_c, 100)
V_super = np.zeros_like(I_super)

# Normal branch (simplified RSJ model)
I_normal_pos = np.linspace(I_c, 3*I_c, 100)
V_normal_pos = (I_normal_pos - I_c) * R_n + 0.1e-3  # Small offset for visibility

I_normal_neg = np.linspace(-3*I_c, -I_c, 100)
V_normal_neg = (I_normal_neg + I_c) * R_n - 0.1e-3

ax1.plot(V_super*1e3, I_super*1e6, 'b-', linewidth=2, label='Supercurrent')
ax1.plot(V_normal_pos*1e3, I_normal_pos*1e6, 'r-', linewidth=2, label='Normal state')
ax1.plot(V_normal_neg*1e3, I_normal_neg*1e6, 'r-', linewidth=2)

ax1.axhline(I_c*1e6, color='gray', linestyle='--', alpha=0.5)
ax1.axhline(-I_c*1e6, color='gray', linestyle='--', alpha=0.5)
ax1.text(0.5, I_c*1e6 + 20, f'I_c = {I_c*1e6:.0f} µA', fontsize=10)

ax1.set_xlabel('Voltage (mV)', fontsize=12)
ax1.set_ylabel('Current (µA)', fontsize=12)
ax1.set_title('Junction I-V Characteristic', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.set_xlim(-2, 2)
ax1.set_ylim(-700, 700)

# Right: I_c variation across wafer (simulated)
np.random.seed(42)
n_junctions = 50
I_c_measured = I_c * (1 + 0.05 * np.random.randn(n_junctions))  # 5% variation

ax2.hist(I_c_measured*1e6, bins=15, color=COLORS['primary'], edgecolor='black', alpha=0.7)
ax2.axvline(I_c*1e6, color='red', linestyle='--', linewidth=2, label=f'Target: {I_c*1e6:.0f} µA')
ax2.axvline(np.mean(I_c_measured)*1e6, color='green', linestyle='-', linewidth=2, 
            label=f'Mean: {np.mean(I_c_measured)*1e6:.1f} µA')

ax2.set_xlabel('Critical Current (µA)', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title(f'I_c Distribution (σ/μ = {np.std(I_c_measured)/np.mean(I_c_measured)*100:.1f}%)', 
              fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"I_c × R_n = {I_c * R_n * 1e3:.2f} mV (target: 1-2 mV for Nb/AlOx/Nb)")

### DC Test Structures

Include these on every chip for process monitoring:

| Structure | Purpose | Measurement |
|-----------|---------|-------------|
| **Single JJ** | I_c, R_n extraction | I-V curve |
| **JJ array (series)** | I_c uniformity | I-V shows weakest junction |
| **Sheet resistance** | Verify metal layers | 4-point probe |
| **Contact resistance** | Via/contact quality | Kelvin structure |
| **Inductance test** | L/square verification | SQUID or resonance |

### Measurement Best Practices

1. **Use 4-wire sensing** — Eliminates lead resistance errors
2. **Current bias, not voltage bias** — JJs are current-controlled devices
3. **Filter inputs** — Prevent noise-induced switching
4. **Ramp slowly** — Avoid transient heating
5. **Multiple sweeps** — Check for hysteresis and repeatability

---
<a id="3-margin-testing"></a>
# 3. Margin Testing
---

**Operating margins** quantify how robust a circuit is to parameter variations. This is the most critical metric for circuit functionality.

### What Are Margins?

Margins measure the range over which a bias parameter can vary while the circuit still operates correctly:

$$\text{Margin} = \frac{X_{max} - X_{min}}{X_{nominal}} \times 100\%$$

where X is the bias parameter (current, voltage, or excitation amplitude).

### Types of Margins

| Margin Type | Typical Target | What It Tests |
|-------------|----------------|---------------|
| **Bias current** | ±15-25% | DC operating point |
| **Excitation amplitude** (AQFP) | ±20-25% | Clock/power delivery |
| **Clock timing** | ±10-20% of period | Setup/hold times |
| **Input amplitude** | ±15-20% | Signal integrity |
| **Temperature** | ±0.5-1 K | Thermal stability |

In [ ]:
# Visualize margin measurement
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: 1D margin plot
bias = np.linspace(0.5, 1.5, 200)
# Simulated bit error rate vs bias
ber = np.where((bias > 0.75) & (bias < 1.25), 0, 
               np.exp(-50*(np.minimum(np.abs(bias-0.75), np.abs(bias-1.25)))))

ax1.semilogy(bias, ber + 1e-12, 'b-', linewidth=2)
ax1.axvline(0.75, color='red', linestyle='--', linewidth=2)
ax1.axvline(1.25, color='red', linestyle='--', linewidth=2)
ax1.axvline(1.0, color='green', linestyle='-', linewidth=2, label='Nominal')

ax1.axhspan(1e-12, 1e-9, alpha=0.2, color='green', label='Operating region')
ax1.annotate('', xy=(0.75, 1e-6), xytext=(1.25, 1e-6),
             arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax1.text(1.0, 3e-6, '±25% margin', ha='center', fontsize=11, color='red', fontweight='bold')

ax1.set_xlabel('Normalized Bias', fontsize=12)
ax1.set_ylabel('Bit Error Rate', fontsize=12)
ax1.set_title('1D Margin Measurement', fontsize=14, fontweight='bold')
ax1.set_ylim(1e-12, 1)
ax1.set_xlim(0.5, 1.5)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Right: 2D margin map (shmoo plot)
x = np.linspace(0.6, 1.4, 50)
y = np.linspace(0.6, 1.4, 50)
X, Y = np.meshgrid(x, y)

# Operating region is roughly elliptical
Z = ((X - 1.0)**2 / 0.2**2 + (Y - 1.0)**2 / 0.22**2) < 1

ax2.contourf(X, Y, Z.astype(float), levels=[0.5, 1.5], colors=[COLORS['success']], alpha=0.5)
ax2.contour(X, Y, Z.astype(float), levels=[0.5], colors=['green'], linewidths=2)
ax2.plot(1.0, 1.0, 'k*', markersize=15, label='Nominal')

# Draw margin box
rect = plt.Rectangle((0.8, 0.78), 0.4, 0.44, fill=False, edgecolor='red', 
                       linestyle='--', linewidth=2, label='±20% margins')
ax2.add_patch(rect)

ax2.set_xlabel('Bias 1 (normalized)', fontsize=12)
ax2.set_ylabel('Bias 2 (normalized)', fontsize=12)
ax2.set_title('2D Margin Map (Shmoo Plot)', fontsize=14, fontweight='bold')
ax2.set_aspect('equal')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Green region = circuit operates correctly")
print("Goal: Operating region should fully contain the ±20% margin box")

### Margin Measurement Procedure

**1D Margin Sweep:**
1. Set all biases to nominal
2. Apply test pattern (e.g., PRBS, walking ones)
3. Sweep one bias from low to high
4. Record pass/fail at each point
5. Find lower and upper boundaries
6. Repeat for each bias

**2D Margin Map (Shmoo):**
1. Choose two critical biases
2. Sweep both in a grid pattern
3. Record pass/fail at each (x, y) point
4. Visualize as a 2D "shmoo plot"
5. Reveals correlations between parameters

### Interpreting Margin Maps

| Pattern | Interpretation | Action |
|---------|---------------|--------|
| Large, centered region | Good design | Ship it |
| Small region | Marginal design | Redesign or screen |
| Off-center region | Nominal values wrong | Re-target biases |
| Irregular shape | Multiple failure modes | Investigate edges |
| Holes in region | Intermittent failures | Check for noise, flux |

---
<a id="4-functional-testing"></a>
# 4. Functional Testing
---

Functional testing verifies that the circuit performs its intended logic operation.

### Test Pattern Types

| Pattern | Purpose | Coverage |
|---------|---------|----------|
| **Exhaustive** | All input combinations | 100% (small circuits only) |
| **Walking ones/zeros** | Stuck-at faults | Basic connectivity |
| **PRBS** | Random patterns | Statistical coverage |
| **ATPG** (Auto Test Pattern Gen) | Targeted fault coverage | Specific faults |
| **Functional vectors** | Real use cases | Application-specific |

### Test Architecture

For complex circuits, build in test infrastructure:

```
                    ┌─────────────────────────────────────┐
                    │              Chip                   │
    Test           │                                     │
    Input ────────►│  ┌─────────┐      ┌─────────┐      │
                    │  │  Input  │      │ Output  │      │────► Test
                    │  │  Shift  │─────►│  Shift  │      │      Output
                    │  │  Reg    │      │  Reg    │      │
                    │  └────┬────┘      └────▲────┘      │
                    │       │                │           │
                    │       ▼                │           │
                    │  ┌─────────────────────┴─────┐     │
                    │  │                           │     │
                    │  │     Circuit Under Test    │     │
                    │  │                           │     │
                    │  └───────────────────────────┘     │
                    │                                     │
                    └─────────────────────────────────────┘
```

**Shift registers** at I/O allow serial loading/unloading of test vectors, reducing pin count.

### Speed Considerations

| Test Mode | Speed | Purpose |
|-----------|-------|--------|
| **Low-speed scan** | kHz-MHz | Debug, pattern loading |
| **At-speed functional** | GHz | Verify timing margins |
| **Overdrive** | >nominal | Stress testing |

### Pass/Fail Criteria

- **Bit Error Rate (BER)**: Typically require BER < 10⁻¹² for production
- **Pattern match**: All output bits match expected values
- **Timing**: Outputs arrive within specified window

---
<a id="5-speed-timing"></a>
# 5. Speed and Timing
---

### Maximum Frequency Testing

To find the maximum operating frequency:

1. Start at a known-good low frequency
2. Run functional test pattern
3. Increment frequency
4. Repeat until errors appear
5. Back off to find reliable maximum

### Timing Measurements

| Measurement | Method | Precision |
|-------------|--------|----------|
| **Propagation delay** | Input-to-output time | ~10 ps |
| **Setup time** | Clock-to-data timing | ~10 ps |
| **Hold time** | Data-to-clock timing | ~10 ps |
| **Jitter** | Edge-to-edge variation | ~1 ps RMS |

### AQFP Timing Characteristics

In AQFP, timing is determined by the **external AC excitation**:

- **Clock frequency** = excitation frequency (typically 5-10 GHz)
- **Stage delay** = 1/4 clock period (one phase step)
- **Pipeline latency** = N stages × stage delay

Timing extraction approach:
1. Measure output vs. excitation phase relationship
2. Sweep excitation frequency to find margins
3. Characterize setup/hold by shifting input timing relative to excitation

### Frequency Margin Testing

```
    Error Rate
         │
    10⁰  │▓▓▓                              ▓▓▓
         │  ▓▓                            ▓▓
    10⁻⁶ │    ▓▓                        ▓▓
         │      ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    10⁻¹²│        │                    │
         └────────┼────────────────────┼────► Frequency
                f_min              f_max
                  
                  ◄─── Operating Range ───►
```

The operating frequency range reveals timing margins in the design.

---
<a id="6-test-structures"></a>
# 6. On-Chip Test Structures
---

Every chip should include **Process Control Monitors (PCMs)** to diagnose issues.

### Essential Test Structures

| Structure | What It Measures | Failure Indication |
|-----------|------------------|-------------------|
| **Single JJ (multiple sizes)** | I_c, R_n vs. area | Process variation |
| **JJ chain (series)** | I_c uniformity | Weak junction in chain |
| **Sheet R (each metal)** | Layer resistance | Deposition issues |
| **Via chain** | Contact resistance | Etch/clean problems |
| **Inductor (known L)** | Inductance | Layer thickness |
| **SQUID magnetometer** | Background flux | Flux trapping |
| **Margin monitor** | Single gate margins | Process centering |
| **Buffer chain** | Stage delay, signal integrity | Timing verification |

### Test Structure Placement

```
    ┌─────────────────────────────────────────────────┐
    │ PCM │           Main Circuit            │ PCM  │
    │     │                                   │      │
    ├─────┤                                   ├──────┤
    │     │                                   │      │
    │     │                                   │      │
    │     │                                   │      │
    │     │                                   │      │
    ├─────┤                                   ├──────┤
    │ PCM │                                   │ PCM  │
    └─────────────────────────────────────────────────┘
    
    PCMs at corners and edges detect gradient across chip
```

### SQUID-Based Flux Monitor

A simple DC SQUID can detect trapped flux:

- Measure I_c modulation vs. applied field
- Offset from expected curve indicates trapped flux
- Place multiple SQUIDs across chip to map flux distribution

In [ ]:
# Visualize SQUID flux detection
fig, ax = plt.subplots(figsize=(10, 5))

# SQUID I_c vs flux
phi = np.linspace(-3, 3, 500)  # flux in units of Φ₀
I_c_squid = np.abs(np.cos(np.pi * phi))  # Idealized SQUID modulation

# Without trapped flux (centered)
ax.plot(phi, I_c_squid, 'b-', linewidth=2, label='No trapped flux')

# With trapped flux (shifted)
phi_trapped = 0.3  # 0.3 Φ₀ trapped
ax.plot(phi, np.abs(np.cos(np.pi * (phi - phi_trapped))), 'r--', linewidth=2, 
        label=f'Trapped flux = {phi_trapped} Φ₀')

ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(phi_trapped, color='red', linestyle=':', alpha=0.5)

ax.annotate('', xy=(phi_trapped, 0.5), xytext=(0, 0.5),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(phi_trapped/2, 0.55, 'Shift = trapped Φ', ha='center', fontsize=11, color='red')

ax.set_xlabel('Applied Flux (Φ/Φ₀)', fontsize=12)
ax.set_ylabel('Normalized I_c', fontsize=12)
ax.set_title('SQUID Flux Detection', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-3, 3)

plt.tight_layout()
plt.show()

print("The shift in the SQUID modulation curve reveals trapped flux.")
print("Multiple SQUIDs across the chip can map the flux distribution.")

---
<a id="7-failure-modes"></a>
# 7. Common Failure Modes
---

Understanding failure modes helps focus debug efforts.

### Fabrication-Related Failures

| Failure | Symptom | Root Cause |
|---------|---------|------------|
| **Open junction** | No I_c, high R | Barrier too thick, lithography |
| **Shorted junction** | No R_n, low R | Barrier pinholes, particles |
| **I_c too low** | Reduced margins | Barrier thick, oxidation |
| **I_c too high** | Reduced margins, excess power | Barrier thin |
| **Open via** | No connectivity | Etch residue, undercut |
| **Metal short** | Unintended connection | Particles, lithography |
| **Metal open** | No connectivity | Over-etch, particles |

### Design-Related Failures

| Failure | Symptom | Root Cause |
|---------|---------|------------|
| **Path imbalance** | Wrong output values | Unequal delays to gate |
| **Insufficient margin** | Intermittent errors | Undersized JJs or L |
| **Crosstalk** | Pattern-dependent errors | Inadequate spacing |
| **Missing splitter** | Fan-out failure | Forgot fan-out > 1 |
| **Wrong polarity** | Inverted logic | Transformer winding error |

### Environment-Related Failures

| Failure | Symptom | Root Cause |
|---------|---------|------------|
| **Flux trapping** | Random errors per cooldown | Ambient field, shielding |
| **Temperature drift** | Margin variation | Cryostat instability |
| **EMI pickup** | Burst errors | Inadequate filtering |
| **Ground loops** | Offset, noise | Wiring topology |

---
<a id="8-debug-strategies"></a>
# 8. Debug Strategies
---

Systematic debug is essential when circuits fail.

### Debug Flowchart

```
    Circuit fails
         │
         ▼
    ┌─────────────┐     No      ┌─────────────────┐
    │ PCMs pass?  │────────────►│ Fabrication     │
    └──────┬──────┘             │ issue - check   │
           │ Yes                │ wafer lot       │
           ▼                    └─────────────────┘
    ┌─────────────┐     No      ┌─────────────────┐
    │ DC biases   │────────────►│ Setup issue -   │
    │ correct?    │             │ check wiring    │
    └──────┬──────┘             └─────────────────┘
           │ Yes
           ▼
    ┌─────────────┐     No      ┌─────────────────┐
    │ Margins     │────────────►│ Marginal design │
    │ centered?   │             │ or process edge │
    └──────┬──────┘             └─────────────────┘
           │ Yes
           ▼
    ┌─────────────┐     No      ┌─────────────────┐
    │ Repeatable  │────────────►│ Flux trapping   │
    │ across      │             │ or intermittent │
    │ cooldowns?  │             │ connection      │
    └──────┬──────┘             └─────────────────┘
           │ Yes
           ▼
    ┌─────────────────────────────────────────────┐
    │ Pattern-dependent failure - check design   │
    │ (path balance, fan-out, polarity)          │
    └─────────────────────────────────────────────┘
```

### Isolation Techniques

1. **Binary search**: Test half the circuit, narrow down
2. **Known-good substitution**: Replace suspected block
3. **Boundary scan**: Observe internal nodes via shift registers
4. **Bias walking**: Vary one bias while monitoring output
5. **Pattern reduction**: Simplify test pattern to isolate failure

### Correlation Analysis

When failures occur:
- **Lot correlation**: Same wafer lot? → Fab issue
- **Position correlation**: Same die location? → Mask defect
- **Cooldown correlation**: Random per cooldown? → Flux trapping
- **Pattern correlation**: Same input pattern? → Design issue
- **Time correlation**: Drift over hours? → Thermal or bias drift

---
<a id="9-yield-statistics"></a>
# 9. Yield and Statistics
---

### Yield Models

Circuit yield follows the Poisson defect model:

$$Y = e^{-D \cdot N}$$

where:
- Y = yield (probability of working circuit)
- D = defect density per junction
- N = number of junctions

| Defect Density | 1,000 JJ | 10,000 JJ | 100,000 JJ | 1,000,000 JJ |
|----------------|----------|-----------|------------|---------------|
| D = 10⁻⁴ | 90% | 37% | 0.005% | ~0% |
| D = 10⁻⁵ | 99% | 90% | 37% | 0.005% |
| D = 10⁻⁶ | 99.9% | 99% | 90% | 37% |
| D = 3×10⁻⁶ (MIT LL) | 99.7% | 97% | 74% | 5% |

In [ ]:
# Yield vs circuit size for different defect densities
fig, ax = plt.subplots(figsize=(10, 6))

N = np.logspace(2, 7, 100)  # 100 to 10M junctions
defect_densities = [1e-4, 1e-5, 3e-6, 1e-6]
colors = [COLORS['danger'], COLORS['secondary'], COLORS['primary'], COLORS['success']]
labels = ['D = 10⁻⁴', 'D = 10⁻⁵', 'D = 3×10⁻⁶ (MIT LL)', 'D = 10⁻⁶']

for D, color, label in zip(defect_densities, colors, labels):
    Y = np.exp(-D * N) * 100
    ax.semilogx(N, Y, color=color, linewidth=2, label=label)

ax.axhline(50, color='gray', linestyle='--', alpha=0.5)
ax.text(1e7, 52, '50% yield', fontsize=10, color='gray')

# Mark MIT LL 809k demonstration
ax.axvline(809120, color='purple', linestyle=':', alpha=0.7)
ax.text(809120, 85, 'MIT LL\n809k JJ', ha='center', fontsize=9, color='purple')

ax.set_xlabel('Number of Josephson Junctions', fontsize=12)
ax.set_ylabel('Yield (%)', fontsize=12)
ax.set_title('Circuit Yield vs. Size', fontsize=14, fontweight='bold')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_xlim(100, 1e7)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("Yield improvement requires reducing defect density D.")
print("Current state-of-the-art (MIT LL): D ≈ 3×10⁻⁶")
print("Target for million-junction circuits: D < 10⁻⁶")

### Statistical Sampling

For production testing, statistical methods are essential:

| Method | Sample Size | Purpose |
|--------|-------------|--------|
| **100% test** | All | High-value parts |
| **Lot sampling** | n per lot | Process monitoring |
| **SPC (Statistical Process Control)** | Continuous | Trend detection |
| **DOE (Design of Experiments)** | Structured | Process optimization |

### Yield Learning

Track these metrics over time to drive improvement:

1. **First-pass yield**: % working on first test
2. **Final yield**: % working after debug/rework
3. **Yield by failure mode**: Which defects dominate?
4. **Yield vs. circuit size**: Extracting defect density
5. **Yield vs. wafer position**: Edge effects, gradients

---
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); color: white; padding: 30px; margin: 20px -10px -10px -10px; border-radius: 15px 15px 0 0; text-align: center;">

## Summary

### Test Environment
- 4.2 K cryogenic system with magnetic shielding
- Careful thermal and signal integrity management

### Key Measurements
- **DC characterization**: I-V curves, I_c, R_n extraction
- **Margin testing**: ±20-25% operating margins target
- **Functional testing**: Pattern-based verification
- **Speed testing**: Maximum frequency, timing margins

### Test Structures
- Include PCMs on every chip: single JJ, chains, sheet R, SQUIDs
- Buffer chains for timing verification
- Flux monitors for trapped flux detection

### Debug Approach
- Systematic: PCMs → biases → margins → repeatability → patterns
- Correlation analysis: lot, position, cooldown, pattern

### Yield
- Y = e^(-D×N): Defect density determines scaling limit
- Current state-of-the-art: D ≈ 3×10⁻⁶ (MIT LL)

</div>

---
---

<div style="background: linear-gradient(135deg, #16213e 0%, #1a1a2e 100%); color: white; padding: 40px; margin: 20px -10px 20px -10px; border-radius: 15px;">
<h1 style="margin: 0; font-size: 2.2em; text-align: center;">Part 2: Test Equipment Construction</h1>
<p style="margin: 10px 0 0 0; font-size: 1.1em; opacity: 0.9; text-align: center;">Building reliable cryogenic test systems</p>
</div>

## Part 2 Contents

10. [LHe Dip Probe Design](#10-lhe-probe)
11. [Rackmount Cryocooler Systems](#11-cryocooler-systems)
12. [Thermal Design](#12-thermal-design)
13. [Signal Integrity](#13-signal-integrity)
14. [Grounding and Shielding](#14-grounding-shielding)
15. [Practical Wiring](#15-practical-wiring)
16. [Magnetic Shielding](#16-magnetic-shielding)
17. [System Integration Checklist](#17-integration-checklist)

---
<a id="10-lhe-probe"></a>
# 10. LHe Dip Probe Design
---

A **liquid helium dip probe** is the simplest cryogenic test platform — just lower it into a dewar of LHe. Ideal for quick characterization and debug.

### Basic Dip Probe Architecture

```
    Room Temperature Flange
    ════════════════════════════════════════
         │    │    │    │    │    │
         │    │    │    │    │    └── Pump-out port
         │    │    │    │    └─────── Feedthroughs (SMA, DC)
         │    │    │    └──────────── Fill/vent tube
         │    │    └───────────────── Support tube (SS)
         │    │
         │    │         ┌─────────────────┐
         │    │         │  Radiation      │  ← 77 K baffle
         │    │         │  Baffle         │    (reflects IR)
         │    │         └─────────────────┘
         │    │
         │    │    ════════════════════════  ← LHe surface
         │    │              4.2 K
         │    │
         │    │         ┌─────────────────┐
         │    │         │  Sample Stage   │
         │    │         │  ┌───────────┐  │
         │    │         │  │    DUT    │  │
         │    │         │  └───────────┘  │
         │    │         │  (thermalized)  │
         │    │         └─────────────────┘
         │    │
    ─────┴────┴─────────────────────────────
              Dewar Bottom
```

### Bill of Materials

| Component | Material | Purpose |
|-----------|----------|---------|
| **Support tube** | 304 SS, thin wall (0.010") | Low thermal conductivity |
| **Top flange** | 304 SS or Al | Feedthrough mounting |
| **Sample stage** | OFHC Cu | Thermalization |
| **Radiation baffle** | Al or Cu, gold-plated | Block 300K radiation |
| **Coax** | SS or CuNi outer | Signal + low heat leak |
| **DC wiring** | Phosphor bronze, manganin | Low thermal conductivity |
| **Connectors** | SMA (hermetic) | RF feedthrough |

### Key Design Rules

**Thermal:**
- Use thin-wall stainless tube (low k, high strength)
- Thermally anchor all wiring at 77K baffle and 4K stage
- Gold-plate copper parts to reduce emissivity
- Include radiation baffles to block room-temp IR

**Mechanical:**
- Design for ~1 m length (typical dewar depth)
- Support tube OD must fit dewar neck (typically 2-4")
- Include handling features (don't drop it in the dewar!)

**Electrical:**
- All feedthroughs at top flange (room temp)
- Hermetic seals for vacuum-jacketed dewars
- Consider filtered feedthroughs for noise-sensitive measurements

### Sample Stage Design

```
    ┌─────────────────────────────────────────────┐
    │              OFHC Copper Block              │
    │  ┌───────────────────────────────────────┐  │
    │  │         DUT (chip carrier)            │  │
    │  │    ┌─────────────────────────────┐    │  │
    │  │    │          Chip               │    │  │
    │  │    │   ┌─────┐  ┌─────┐         │    │  │
    │  │    │   │ PCM │  │ DUT │         │    │  │
    │  │    │   └─────┘  └─────┘         │    │  │
    │  │    └─────────────────────────────┘    │  │
    │  └───────────────────────────────────────┘  │
    │                                             │
    │  Thermometer ──►  ●                         │
    │  Heater ────────► ▓▓▓                       │
    │                                             │
    │  ════════════════════════════════════════   │
    │       Thermal link to LHe bath              │
    └─────────────────────────────────────────────┘
```

**Critical features:**
- **OFHC copper** (not just "copper") for good thermal conductivity at 4K
- **Thermometer** (Cernox, Si diode, or RuO₂) to verify temperature
- **Heater** (optional) for temperature control
- **Thermal link** to bath — bolt directly to support tube bottom or use copper braid

### Pros and Cons

| Advantage | Disadvantage |
|-----------|--------------|
| Simple, cheap to build | LHe consumable cost |
| Fast cooldown (~30 min) | Limited hold time (hours) |
| Easy access (just pull out) | No temperature control without heater |
| Multiple probes per dewar | Manual operation |
| Excellent base temperature | Not suitable for production |

---
<a id="11-cryocooler-systems"></a>
# 11. Rackmount Cryocooler Systems
---

For production testing and automated measurement, a **closed-cycle cryocooler** eliminates LHe consumables and enables continuous operation.

### System Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          RACKMOUNT TEST SYSTEM                              │
│                                                                             │
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐             │
│  │  Control PC     │  │  Instruments    │  │  Power/Bias     │             │
│  │                 │  │  - Pattern Gen  │  │  - DC Supplies  │             │
│  │  - Automation   │  │  - Scope/BER    │  │  - AC Excite    │             │
│  │  - Data Acq     │  │  - Counters     │  │  - Heater Ctrl  │             │
│  └────────┬────────┘  └────────┬────────┘  └────────┬────────┘             │
│           │                    │                    │                       │
│           └────────────────────┼────────────────────┘                       │
│                                │                                            │
│  ══════════════════════════════╪════════════════════════════════════════   │
│           Patch Panel / Breakout (room-temp interface)                      │
│  ══════════════════════════════╪════════════════════════════════════════   │
│                                │                                            │
│                     ┌──────────┴──────────┐                                │
│                     │   Vacuum Chamber    │                                │
│                     │   ┌──────────────┐  │                                │
│                     │   │   40-50 K    │  │  ← 1st stage                   │
│                     │   │   Shield     │  │                                │
│                     │   │  ┌────────┐  │  │                                │
│                     │   │  │  4 K   │  │  │  ← 2nd stage                   │
│                     │   │  │ Stage  │  │  │                                │
│                     │   │  │ [DUT]  │  │  │                                │
│                     │   │  └────────┘  │  │                                │
│                     │   └──────────────┘  │                                │
│                     └─────────┬───────────┘                                │
│                               │                                            │
│                     ┌─────────┴───────────┐                                │
│                     │   Cryocooler Head   │                                │
│                     │   (Gifford-McMahon  │                                │
│                     │    or Pulse Tube)   │                                │
│                     └─────────┬───────────┘                                │
│                               │                                            │
│                     ┌─────────┴───────────┐                                │
│                     │   Compressor Unit   │  ← Often floor-mounted         │
│                     │   (water cooled)    │                                │
│                     └─────────────────────┘                                │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Cryocooler Selection

| Type | 4K Cooling Power | Vibration | Cost | Notes |
|------|------------------|-----------|------|-------|
| **Gifford-McMahon (GM)** | 1-2 W | Higher | $$ | Reliable, common |
| **Pulse Tube (PT)** | 0.5-1.5 W | Lower | $$$ | Less maintenance |
| **Stirling** | 0.1-0.5 W | Medium | $ | Smaller capacity |

**Common models:**
- Sumitomo RDK-408D2 (GM, 1.5W @ 4.2K)
- Cryomech PT410 (PT, 1.0W @ 4.2K)
- Sumitomo RP-082B2 (PT, 0.9W @ 4.2K)

### Vacuum Chamber Design

**Chamber requirements:**
- Base pressure: <10⁻⁵ torr (turbo + roughing pump)
- Feedthrough ports for RF, DC, optical
- Viewport (optional) for alignment
- Helium leak rate: <10⁻⁹ mbar·L/s

```
    ┌────────────────────────────────────────────────────┐
    │                   Top Plate                        │
    │   ┌────┐  ┌────┐  ┌────┐  ┌────┐  ┌────┐         │
    │   │SMA │  │SMA │  │ DC │  │ DC │  │Opt │         │
    │   │ 1  │  │ 2  │  │ 1  │  │ 2  │  │Fib │         │
    │   └──┬─┘  └──┬─┘  └──┬─┘  └──┬─┘  └──┬─┘         │
    │      │       │       │       │       │            │
    │  ════╪═══════╪═══════╪═══════╪═══════╪════════   │
    │      │       │       │       │       │            │
    │      │       │       │       │       │            │
    │   ┌──┴───────┴───────┴───────┴───────┴──┐        │
    │   │          40 K Radiation Shield       │        │
    │   │  ┌───────────────────────────────┐   │        │
    │   │  │         4 K Sample Stage      │   │        │
    │   │  │                               │   │        │
    │   │  │   [====== DUT PCB ======]     │   │        │
    │   │  │                               │   │        │
    │   │  └───────────────────────────────┘   │        │
    │   └──────────────────────────────────────┘        │
    │                      │                            │
    │                 Cryocooler                        │
    │                   Cold Head                       │
    └──────────────────────┴────────────────────────────┘
```

### Thermal Budget

At 4K, you have limited cooling power. Every milliwatt counts:

| Heat Source | Typical Load | Mitigation |
|-------------|--------------|------------|
| **Conduction (wiring)** | 1-10 mW/coax | Use SS or CuNi, thermal intercept |
| **Radiation** | 10-100 mW | MLI blankets, 40K shield |
| **Joule heating (DUT)** | 0.1-10 mW | Design-dependent |
| **AC excitation** | 1-10 mW | Efficient coupling |
| **Thermal switches** | Variable | Minimize contact resistance |

**Rule of thumb**: Keep total 4K load under 50% of rated cooling power for margin.

### Thermal Intercept Strategy

Intercept heat at higher temperatures where cooling is cheaper:

```
    300 K (Room Temp)
      │
      │  High-R wiring (SS, CuNi)
      │
      ▼
    ═══════════════════════════════  ← 40-50 K intercept
      │                                 (1st stage: 30-50 W available)
      │  Thermalized at 40K stage
      │
      │  Low-R wiring below here OK (Cu, NbTi)
      │
      ▼
    ═══════════════════════════════  ← 4 K stage
                                       (2nd stage: 1-2 W available)
      │
      ▼
     DUT
```

**Intercept everything at 40K:**
- All coax outer conductors
- DC wiring (wrapped on bobbin)
- Optical fibers
- Structural supports

---
<a id="12-thermal-design"></a>
# 12. Thermal Design
---

Thermal management determines whether your system reaches base temperature and how much margin you have.

### Thermal Conductivity at Cryogenic Temperatures

Material properties change dramatically at low T:

| Material | k at 300K (W/m·K) | k at 4K (W/m·K) | Use |
|----------|-------------------|-----------------|-----|
| **OFHC Copper** | 400 | 200-2000 (RRR dependent) | Heat sinks, straps |
| **304 Stainless** | 15 | 0.3 | Low-k supports, wiring |
| **G-10 Fiberglass** | 0.3 | 0.05 | Structural standoffs |
| **Vespel SP-22** | 0.35 | 0.02 | Low-k precision parts |
| **Sapphire** | 40 | 10 | Electrical isolation + thermal |
| **Manganin** | 22 | 5 | Low thermal conductivity wire |

**Key insight**: Stainless steel k drops 50× from 300K to 4K — great for thermal isolation.

### Heat Load Calculation

**Conduction along a rod:**
$$\dot{Q} = \frac{A}{L} \int_{T_c}^{T_h} k(T) \, dT$$

For practical calculation, use thermal conductivity integrals:

| Material | ∫k dT (4K to 300K) | Units |
|----------|-------------------|-------|
| **304 SS** | 1,050 | W/m |
| **Phosphor Bronze** | 950 | W/m |
| **Manganin** | 3,000 | W/m |
| **OFHC Cu (RRR=50)** | 60,000 | W/m |

**Example**: 0.5 mm dia × 0.5 m long stainless coax outer:
$$\dot{Q} = \frac{\pi (0.25\text{mm})^2}{500\text{mm}} \times 1050 \frac{\text{W}}{\text{m}} = 0.4 \text{ mW}$$

### Sample Stage Thermalization

The DUT must be at 4K, not floating above it:

```
    ┌─────────────────────────────────────────────────────────────┐
    │                   4K Cryocooler Stage                       │
    │   (OFHC Cu, bolted to cold head)                           │
    │                         │                                   │
    │                    ┌────┴────┐                              │
    │                    │  Cu     │ ← Flexible thermal link      │
    │                    │  Braid  │   (allows alignment)         │
    │                    └────┬────┘                              │
    │                         │                                   │
    │              ┌──────────┴──────────┐                        │
    │              │   Sample Stage      │                        │
    │              │   (OFHC Cu block)   │                        │
    │              │         │           │                        │
    │              │    ┌────┴────┐      │                        │
    │              │    │ Carrier │      │ ← PCB or Cu carrier    │
    │              │    │   PCB   │      │                        │
    │              │    │  [DUT]  │      │ ← Chip bonded/mounted  │
    │              │    └─────────┘      │                        │
    │              └─────────────────────┘                        │
    └─────────────────────────────────────────────────────────────┘
```

**Critical thermal interfaces:**

| Interface | Contact Method | Thermal Resistance |
|-----------|----------------|-------------------|
| Cold head → Stage | Bolted, gold-plated | <0.1 K/W |
| Stage → Carrier | Bolted + grease or indium | <1 K/W |
| Carrier → Chip | Epoxy, grease, or varnish | ~1-10 K/W |

**Tips:**
- Use **Apiezon N grease** for demountable joints
- Use **indium foil** for permanent low-R joints
- **Gold plate** all copper surfaces (prevents oxidation)
- **Bolt torque matters** — too loose = high R, too tight = cracks

### Radiation Shielding

300K blackbody radiation delivers ~450 W/m² to a 4K surface. Shield it:

**Multi-Layer Insulation (MLI):**
- Alternating aluminized mylar + netting
- 10-30 layers typical
- Reduces radiation by 100-1000×

**40K Radiation Shield:**
- Copper or aluminum, gold-plated
- Attached to 1st stage (40-50K)
- Blocks direct 300K line-of-sight to 4K

```
    300K Chamber Wall
         │
         │   ← MLI blankets
         │
    ─────┼─────  40K Shield (attached to 1st stage)
         │
         │   ← MLI or open (already cold)
         │
    ─────┼─────  4K Stage
```

### Thermometry

Always verify your temperature:

| Sensor | Range | Accuracy | Notes |
|--------|-------|----------|-------|
| **Si Diode** | 1.4-500 K | ±0.5 K | Standard, cheap |
| **Cernox** | 0.1-420 K | ±0.5% | Low magnetic field sensitivity |
| **RuO₂** | 0.05-40 K | ±2% | Good for <4K |
| **Pt RTD** | 30-800 K | ±0.1 K | Not useful below 30K |

**Placement**: Mount thermometer on same stage as DUT, with good thermal contact.

---
<a id="13-signal-integrity"></a>
# 13. Signal Integrity
---

SFQ pulses are ~2 ps wide and ~1-2 mV amplitude. Preserving signal integrity through 1+ meters of cabling is non-trivial.

### Coaxial Cable Selection

| Cable Type | Z₀ | Attenuation (4K) | Thermal Load | Use Case |
|------------|-----|------------------|--------------|----------|
| **UT-085-SS** | 50Ω | ~1 dB/m @ 10 GHz | ~1 mW | Standard RF |
| **UT-047-SS** | 50Ω | ~2 dB/m @ 10 GHz | ~0.3 mW | Low heat load |
| **SC-086/50-SS-SS** | 50Ω | ~0.5 dB/m @ 10 GHz | ~0.5 mW | Premium SS |
| **SC-086/50-NbTi-NbTi** | 50Ω | ~0.1 dB/m @ 10 GHz | <0.1 mW | Superconducting |
| **SC-086/50-CN-CN** | 50Ω | ~1.5 dB/m @ 10 GHz | ~0.5 mW | CuNi, moderate |

**Rule**: Below T_c, NbTi coax has near-zero loss but is expensive.

### Impedance Matching

SFQ circuits typically interface at **50Ω or matched to PTL impedance (~5Ω internal)**.

```
    Room Temp                    4K Stage
    ──────────────────────────────────────────────────
    
    Pattern Gen    50Ω Coax     On-chip      On-chip
       50Ω ────────────────────► Driver ────► Circuit
                                 (matches
                                  coax to
                                  PTL)
    
       50Ω ◄──────────────────── Receiver ◄── Output
    Readout        50Ω Coax     (matches
                                 PTL to
                                 coax)
```

**Mismatch consequences:**
- Reflections → signal distortion
- Standing waves → frequency-dependent response
- At SFQ pulse speeds, even small reflections cause errors

### Connector Selection

| Connector | Frequency Range | Notes |
|-----------|-----------------|-------|
| **SMA** | DC-18 GHz | Standard, cheap |
| **3.5mm** | DC-34 GHz | Compatible with SMA |
| **2.92mm (K)** | DC-40 GHz | Higher frequency |
| **2.4mm** | DC-50 GHz | Precision, expensive |
| **1.85mm (V)** | DC-67 GHz | Highest performance |

**For SCE testing**: SMA is usually sufficient (5-10 GHz operation).

### Cable Assembly Best Practices

1. **Keep cables matched length** where timing matters
2. **Avoid tight bends** — minimum bend radius = 5× cable OD
3. **Torque connectors properly** — 8 in-lb for SMA
4. **Use semi-rigid where possible** — lower loss than flexible
5. **Label everything** — you will forget which cable goes where

### Attenuation and Filtering

Sometimes you need to attenuate signals entering the cryostat to reduce noise:

```
    300K                  40K                 4K
    ─────────────────────────────────────────────────
         │                 │                  │
         │    20 dB Atten  │    10 dB Atten   │
    ─────┼────[████]───────┼────[████]────────┼────► DUT
         │                 │                  │
```

**Attenuation strategy:**
- Attenuators thermalize coax center conductor
- Johnson noise from 300K attenuator is reduced at 4K
- Total attenuation sets input noise floor

**Low-pass filtering:**
- Block high-frequency noise
- Use reflectionless filters to avoid standing waves
- Mount filters at cold stage for best performance

### DC Line Filtering

DC bias lines pick up RF interference. Filter aggressively:

| Filter Type | Cutoff | Use |
|-------------|--------|-----|
| **RC low-pass** | 1-100 kHz | DC bias |
| **LC low-pass** | 1-100 MHz | Faster signals |
| **π-filter** | 1-100 MHz | Better rejection |
| **Feedthrough capacitor** | 10 MHz+ | At vacuum feedthrough |

**Typical DC line:**
```
    300K:    π-filter (1 MHz cutoff)
    40K:     RC filter (10 kHz cutoff) — thermalizes wire
    4K:      RC filter (1 kHz cutoff) — final noise reduction
```

---
<a id="14-grounding-shielding"></a>
# 14. Grounding and Shielding
---

Ground loops and poor shielding are the #1 cause of unexplained noise in cryogenic measurements.

### The Ground Loop Problem

A ground loop forms when there are multiple paths to ground:

```
    ┌──────────────────────────────────────────────────────────┐
    │                      BAD: Ground Loop                    │
    │                                                          │
    │    Instrument A                      Instrument B        │
    │        │                                  │              │
    │        │    ┌─────────┐                   │              │
    │        └───►│   DUT   │◄──────────────────┘              │
    │             └────┬────┘                                  │
    │                  │                                       │
    │         ┌────────┴────────┐                              │
    │         │                 │                              │
    │         ▼                 ▼                              │
    │    Ground via         Ground via                         │
    │    Instrument A       Instrument B                       │
    │         │                 │                              │
    │         └────────┬────────┘   ← Current flows in loop!  │
    │                  │              Induces voltage in DUT   │
    │             Building Ground                              │
    └──────────────────────────────────────────────────────────┘
```

The loop picks up 60 Hz (or other interference) and induces voltage across the DUT ground.

### Star Ground Topology

Use a single-point (star) ground:

```
    ┌──────────────────────────────────────────────────────────┐
    │                     GOOD: Star Ground                    │
    │                                                          │
    │    Instrument A         Instrument B        Instrument C │
    │        │                     │                   │       │
    │        │                     │                   │       │
    │        └──────────┬──────────┴───────────────────┘       │
    │                   │                                      │
    │                   ▼                                      │
    │              STAR POINT                                  │
    │             (single ground)                              │
    │                   │                                      │
    │                   │                                      │
    │                   ▼                                      │
    │              Building Ground                             │
    │                                                          │
    │    All instruments share ONE path to ground              │
    │    No loops, no circulating currents                     │
    └──────────────────────────────────────────────────────────┘
```

### Practical Implementation

**For a rack system:**

1. **Designate one ground point** — usually the cryostat chassis
2. **Float all instruments** — use isolated power (or accept they're grounded via power cord)
3. **Connect coax shields** at one end only (usually the DUT end)
4. **Use differential signals** where possible

**Cryostat grounding:**
```
    Room Temp Patch Panel
    ══════════════════════════════════════════
         │           │           │
         │           │           │
    Coax │      DC   │      Coax │
    shield      return     shield
         │           │           │
         └─────┬─────┴─────┬─────┘
               │           │
               ▼           │
         4K Stage          │
         Ground Plane ◄────┘
               │
               ▼
         Cryostat Chassis ─────► Building Ground
         (single point)
```

### Common Grounding Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| **Multiple earth grounds** | 60 Hz pickup | Single-point ground |
| **Shield grounded both ends** | Ground loop hum | Ground one end only |
| **Floating cryostat** | Static discharge, noise | Bond chassis to earth |
| **Mixed analog/digital grounds** | Digital noise on analog | Separate, join at star |
| **Long ground wires** | Inductive pickup | Short, wide conductors |

### Shielding

**Electromagnetic shielding:**
- Vacuum chamber acts as Faraday cage
- All feedthroughs must maintain shielding continuity
- Use shielded cables for all signals

**Magnetic shielding:**
- See Section 16 (Magnetic Shielding)
- Essential for flux-sensitive SQUIDs and AQFP circuits

### Isolation Techniques

When grounding is problematic, isolate instead:

| Technique | Use Case | Bandwidth |
|-----------|----------|-----------|
| **Isolation transformer** | AC power | 60 Hz only |
| **Opto-isolator** | Digital control | DC-10 MHz |
| **Fiber optic** | High-speed data | DC-10+ GHz |
| **Balanced differential** | Analog signals | DC-100 MHz |
| **Instrumentation amp** | Low-level signals | DC-1 MHz |

---
<a id="15-practical-wiring"></a>
# 15. Practical Wiring
---

The devil is in the details. Here's how to actually wire a cryogenic test system.

### DC Wiring

**Wire selection for DC bias:**

| Wire | AWG | Resistance | Thermal Load | Use |
|------|-----|------------|--------------|-----|
| **Phosphor bronze** | 32-36 | ~3 Ω/m | ~50 µW | Standard DC |
| **Manganin** | 32-36 | ~10 Ω/m | ~100 µW | Higher R needed |
| **Constantan** | 32-36 | ~15 Ω/m | ~80 µW | Thermocouples |
| **NbTi** | 36 | 0 (below T_c) | ~10 µW | Superconducting |

**Twisted pair configuration:**
```
    Signal (+) ──╲╱──╲╱──╲╱──╲╱──► 
                 ╲  ╱╲  ╱╲  ╱╲  ╱
    Signal (-) ──╱╲──╱╲──╱╲──╱╲──►
    
    Twisting rejects common-mode pickup
    Aim for 1-2 twists per cm
```

**Wiring loom construction:**
1. Bundle wires with PTFE tape or Kevlar thread (not nylon — becomes brittle)
2. Wrap around thermal intercept bobbin at 40K and 4K
3. Secure with GE varnish or Stycast (avoid standard adhesives — outgas)
4. Label both ends!

### Thermal Anchoring Wires

Every wire must be thermalized at each stage:

```
    ┌──────────────────────────────────────────────────────┐
    │              Thermal Intercept Bobbin                │
    │                                                      │
    │     OFHC Cu bobbin, bolted to cold stage            │
    │                                                      │
    │        ┌────────────────────────────────┐           │
    │        │                                │           │
    │   ─────┤  Wind 10-20 turns of wire     ├─────      │
    │        │  around bobbin                 │           │
    │        │                                │           │
    │        └────────────────────────────────┘           │
    │                                                      │
    │     Varnish or Stycast to secure and improve        │
    │     thermal contact                                  │
    └──────────────────────────────────────────────────────┘
```

**Thermal contact options:**
- **GE varnish**: Easy to apply, removable with ethanol
- **Stycast 2850**: Stronger, permanent
- **Clamped**: Best thermal contact but bulky

---

### Kapton Flex / Flex PCB Wiring

**Flex circuits are ideal for cryogenic interconnects** — controlled impedance, high density, low thermal mass, and reproducible.

#### Flex PCB Stack-Up for Cryo

```
    ┌─────────────────────────────────────────────────────┐
    │                 Typical Cryo Flex Stack             │
    │                                                     │
    │    ══════════════════════════════  Coverlay        │
    │    ──────────────────────────────  Cu trace (18µm) │
    │    ══════════════════════════════  Kapton (25-50µm)│
    │    ──────────────────────────────  Cu ground plane │
    │    ══════════════════════════════  Coverlay        │
    │                                                     │
    │    Total thickness: ~100-150 µm                    │
    └─────────────────────────────────────────────────────┘
```

#### Why Flex for Cryo?

| Advantage | Explanation |
|-----------|-------------|
| **Low thermal mass** | Thin materials = fast thermalization |
| **Low thermal conductivity** | Kapton k ~ 0.1 W/m·K at 4K |
| **Controlled impedance** | 50Ω microstrip or stripline |
| **High density** | 100+ traces in narrow ribbon |
| **Reproducible** | Manufactured, not hand-wired |
| **Survives thermal cycling** | Kapton is robust to 4K |

#### Design Rules for Cryo Flex

| Parameter | Recommendation | Notes |
|-----------|----------------|-------|
| **Substrate** | Kapton (polyimide) | Survives cryo cycling |
| **Copper** | Electrodeposited (ED) | More flexible than RA |
| **Trace width** | ≥75 µm (3 mil) | For manufacturability |
| **Spacing** | ≥75 µm (3 mil) | Avoid shorts |
| **Impedance** | 50Ω microstrip | Match to coax |
| **Ground plane** | Continuous | For controlled Z₀ |
| **Stiffeners** | At connector ends | FR-4 or Kapton |
| **Strain relief** | Service loops | Allow contraction |

#### Flex Thermal Anchoring

Flex cables must be thermalized at each stage:

```
    ┌────────────────────────────────────────────────────────────┐
    │                  Flex Cable Thermal Clamp                  │
    │                                                            │
    │    OFHC Cu clamp bar (gold-plated)                        │
    │         ┌──────────────────────────────────┐              │
    │         │▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓│ ← Top clamp   │
    │    ═════╪══════════════════════════════════╪═════ Flex    │
    │         │▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓│ ← Bottom      │
    │         └──────────────────────────────────┘    clamp     │
    │                        │                                   │
    │                        ▼                                   │
    │                   Cold stage                               │
    │                                                            │
    │    Apply Apiezon N grease between flex and clamp          │
    │    Bolt clamp to cold stage                               │
    └────────────────────────────────────────────────────────────┘
```

**Key points:**
- Clamp the ground plane side against copper for best thermal contact
- Use thermal grease (Apiezon N) in the joint
- Service loop between stages allows contraction

#### Flex Connector Options

| Connector | Pitch | Density | Notes |
|-----------|-------|---------|-------|
| **ZIF (Zero Insertion Force)** | 0.5-1.0 mm | High | Easy mate/demate |
| **Samtec SFMC** | 0.635 mm | High | Cryo-rated |
| **Hirose FH12** | 0.5 mm | Very high | Consumer-grade, works at cryo |
| **Solder pads** | Custom | Highest | Permanent connection |
| **Wire bond pads** | 100 µm | Chip-level | Direct to chip |

#### Example: High-Density DC Bias Flex

```
    300K Feedthrough                           4K Sample Stage
    ┌──────────────┐                          ┌──────────────┐
    │ 50-pin ZIF   │                          │ 50-pin ZIF   │
    │ connector    │                          │ connector    │
    └──────┬───────┘                          └──────┬───────┘
           │                                         │
           │      ┌─────────────────────┐            │
           │      │   40K Thermal       │            │
           └──────┤   Clamp            ├────────────┘
                  │   (flex thermalized)│
                  └─────────────────────┘
    
    50 DC bias lines in a single 25mm-wide flex cable
    Thermal load: ~1-2 mW total (much less than 50 wires)
```

#### Common Flex Issues

| Problem | Cause | Fix |
|---------|-------|-----|
| **Cracking at cold** | Tight bend radius | Increase radius, add service loop |
| **Delamination** | Poor adhesion | Use cryo-rated flex vendor |
| **Impedance variation** | Tolerance stack-up | Tighter specs, TDR verify |
| **Connector failure** | Thermal contraction | Use cryo-rated connectors |
| **Ground bounce** | Via inductance | More ground vias |

---

### PCB Design for Cryogenic Use

**Substrate selection:**

| Material | CTE (ppm/K) | Notes |
|----------|-------------|-------|
| **FR-4** | 14-17 | Standard, cheap, OK for carriers |
| **Rogers 4350** | 11 | Better RF, lower loss |
| **Alumina (Al₂O₃)** | 7 | Matches Si, good thermal |
| **AlN** | 4.5 | Excellent thermal, expensive |
| **LTCC** | 5-7 | Multilayer, hermetic |

**CTE matching matters**: Si chip (2.6 ppm/K) on FR-4 (15 ppm/K) = stress during cooldown.

**PCB layout rules:**
- Wide traces for current-carrying paths (thermal conductivity too)
- Via stitching around sensitive signals
- Ground plane on one layer
- No sharp corners (stress concentrators)

### Connectorization

**At the chip level:**
- Wire bonding (25 µm Al or Au)
- Ribbon bonding (wider, lower inductance)
- Flip-chip bumps (advanced)

**At the PCB level:**
- SMP/SMPM: Push-on, good for quick swaps
- SMA: Standard, reliable
- Custom spring contacts: For automated testing

**Connector thermal expansion:**
- SMA works fine at 4K
- Use stainless or BeCu bodies (not brass — contracts too much)
- Retorque after first cooldown

### Cable Management

```
    ┌────────────────────────────────────────────────────────┐
    │                    Cable Organization                  │
    │                                                        │
    │    Top Plate (300K)                                   │
    │    ════════════════════════════════════════════════   │
    │         ║     ║     ║     ║     ║     ║              │
    │         ║     ║     ║     ║     ║     ║              │
    │    ─────╫─────╫─────╫─────╫─────╫─────╫───── 40K     │
    │         ║     ║     ║     ║     ║     ║    Service   │
    │         ╟─────╫─────╫─────╫─────╫─────╫    loop      │
    │         ║     ║     ║     ║     ║     ║              │
    │    ─────╫─────╫─────╫─────╫─────╫─────╫───── 4K      │
    │         ║     ║     ║     ║     ║     ║    Service   │
    │         ╟─────╨─────╨─────╨─────╨─────╫    loop      │
    │         ║                             ║              │
    │    ═════╩═════════════════════════════╩═════ DUT     │
    │                                                        │
    │    Service loops allow thermal contraction            │
    │    without stress on connections                      │
    └────────────────────────────────────────────────────────┘
```

**Service loop rules:**
- Allow 1-2% length for contraction (300K to 4K)
- Secure cables to prevent vibration
- Don't over-constrain — thermal stress breaks things

---
<a id="16-magnetic-shielding"></a>
# 16. Magnetic Shielding
---

Superconducting circuits are extremely sensitive to magnetic fields. Effective shielding is essential.

### Why Shield?

| Effect | Field Sensitivity | Consequence |
|--------|-------------------|-------------|
| **Flux trapping** | ~1 µT during cooldown | Random failures |
| **SQUID offset** | ~1 nT | Measurement error |
| **JJ I_c modulation** | ~10 µT | Margin reduction |
| **AQFP operating point** | ~1-10 µT | Bit errors |

Earth's field is ~50 µT. Lab fields can be much higher near equipment.

### Shielding Materials

| Material | Type | Permeability | Temperature Range | Notes |
|----------|------|--------------|-------------------|-------|
| **Finemet** | Nanocrystalline | 100,000+ | 4-300 K | **Preferred** — no annealing needed |
| **Metglas 2714A** | Amorphous | 80,000-100,000 | >77 K | Good alternative to Finemet |
| **Mu-metal** | High-µ alloy | 20,000-100,000 | >77 K | Requires annealing after forming |
| **Cryoperm 10** | High-µ alloy | 50,000-100,000 | 4-300 K | Cryo-compatible |
| **Pb (Lead)** | Type I SC | Perfect diamagnet | <7.2 K | **Preferred SC shield** — easy to form |
| **Nb** | Type II SC | Perfect diamagnet | <9.2 K | Higher T_c but harder to work |

**Recommendations:**
- **Warm stages**: Finemet (no stress sensitivity, highest µ)
- **Cold SC shield**: Lead foil (easy to form, good T_c margin at 4.2K)

### Multi-Layer Shield Design

**Shielding factor** multiplies with each layer:

```
    ┌───────────────────────────────────────────────────────────┐
    │  External Field: 50 µT (Earth + local)                   │
    │                                                           │
    │  ╔═══════════════════════════════════════════════════╗   │
    │  ║  Finemet (room temp)               ÷100-500       ║   │
    │  ║  Field: ~100-500 nT                               ║   │
    │  ║  ┌─────────────────────────────────────────────┐  ║   │
    │  ║  │  Finemet or Cryoperm (40K)      ÷50-100    │  ║   │
    │  ║  │  Field: ~1-10 nT                           │  ║   │
    │  ║  │  ╔═════════════════════════════════════╗   │  ║   │
    │  ║  │  ║  Lead superconducting (4K)         ║   │  ║   │
    │  ║  │  ║  Field: <0.1 nT (expelled)         ║   │  ║   │
    │  ║  │  ║  ┌───────────────────────────────┐ ║   │  ║   │
    │  ║  │  ║  │            DUT                │ ║   │  ║   │
    │  ║  │  ║  └───────────────────────────────┘ ║   │  ║   │
    │  ║  │  ╚═════════════════════════════════════╝   │  ║   │
    │  ║  └─────────────────────────────────────────────┘  ║   │
    │  ╚═══════════════════════════════════════════════════╝   │
    │                                                           │
    │  Total shielding factor: ~10⁵ - 10⁶                      │
    └───────────────────────────────────────────────────────────┘
```

### Finemet Advantages Over Mu-Metal

| Property | Finemet | Mu-Metal |
|----------|---------|----------|
| **Permeability** | 100,000+ | 20,000-100,000 |
| **Stress sensitivity** | Low | High (needs anneal) |
| **After forming** | Ready to use | Must anneal |
| **Frequency response** | Good to MHz | Good to ~100 kHz |
| **Cost** | Higher | Lower |
| **Availability** | Tape/ribbon form | Sheet, can, custom |

**Finemet is supplied as tape** — wrap multiple layers around your shield can or form a cylinder from wound tape.

### Practical Shield Construction

**Finemet outer shield (room temp):**
- Wind Finemet tape around cylindrical form
- 3-5 layers gives excellent shielding
- No annealing required — just wind and use
- Can also get pre-formed Finemet cans from vendors

**Lead inner shield (4K):**
- 0.5-1 mm thick Pb sheet
- Easy to cut with scissors, form by hand
- Solder seams with Pb-Sn solder (use flux)
- Overlap seams generously (flux enters through gaps)
- T_c = 7.2 K, so good margin at 4.2 K operation

```
    Lead Shield Construction
    
         ┌─────────────────────────┐
         │      Pb cylinder        │
         │    (rolled from sheet)  │
         │                         │
         │   ┌─────────────────┐   │
         │   │    Overlap      │   │
         │   │    seam         │   │
         │   │    (soldered)   │   │
         │   └─────────────────┘   │
         │                         │
         │   ┌─────────────────┐   │
         │   │   Pb end cap    │   │ ← Solder or crimp
         │   └─────────────────┘   │
         └─────────────────────────┘
         
    Minimize holes — every penetration leaks flux
```

### Critical: Cool-Through Procedure

The superconducting shield must transition **in a low-field environment**:

```
    Temperature
         │
   7.2K ─┼───────────────────●──────────────────────  (Pb T_c)
         │                   │
         │    ▲              │  Field expelled
         │    │ Applied      │  (Meissner effect)
         │    │ field        │
         │    │ < 1 µT       ▼
         │                   
    4 K ─┼────────────────────────────────────────────► Time
         │
         ├──────────────────►│
              Cooldown        │
                              │
              Keep external field low during T_c transition
```

**If field is too high during T_c**: Flux traps inside Pb shield, defeating its purpose.

### Degaussing

High-µ shields (Finemet, mu-metal) can become magnetized. Degauss to restore performance:

1. Apply AC field at 50-60 Hz, amplitude > saturation field
2. Slowly reduce amplitude to zero over 30-60 seconds
3. Repeat if shielding factor doesn't improve

**Finemet advantage**: Less susceptible to permanent magnetization than mu-metal.

**When to degauss:**
- After exposure to large magnetic transients
- After shipping/handling
- When shielding performance degrades

### Magnetic Field Sources to Eliminate

| Source | Typical Field | Mitigation |
|--------|---------------|------------|
| **Earth's field** | 30-60 µT | Shielding |
| **Building steel** | 10-100 µT | Distance, shielding |
| **Power lines** | 0.1-10 µT AC | Distance, cancellation |
| **Motors** | 10-1000 µT | Distance, turn off |
| **Ion pumps** | 1-100 µT | Distance, turn off during cooldown |
| **Turbo pumps** | 1-50 µT | Magnetic bearing turbos better |
| **CRT monitors** | 1-10 µT | Use LCD instead |
| **Loudspeakers** | 10-100 µT | Remove from area |
| **Cell phones** | 1-10 µT | Remove from area |

---
<a id="17-integration-checklist"></a>
# 17. System Integration Checklist
---

Use this checklist when building or troubleshooting a cryogenic test system.

### Pre-Assembly

| Item | Check | Notes |
|------|-------|-------|
| □ | All copper parts gold-plated | Prevents oxidation |
| □ | Mu-metal shields annealed | After any machining |
| □ | Coax cables tested | TDR or VNA verify impedance |
| □ | DC wiring resistance measured | Verify no opens/shorts |
| □ | Feedthrough leak checked | <10⁻⁹ mbar·L/s |
| □ | Thermometers calibrated | Or use manufacturer cal |

### Assembly

| Item | Check | Notes |
|------|-------|-------|
| □ | Thermal intercepts installed | At 40K and 4K stages |
| □ | Service loops adequate | 1-2% contraction allowance |
| □ | Grounding star point defined | Document where |
| □ | Coax shields grounded one end | Typically at DUT |
| □ | Magnetic shields installed | Verify no gaps |
| □ | Radiation shields in place | MLI applied |
| □ | Thermometers mounted | Good thermal contact |
| □ | Heater installed (if needed) | For temperature control |

### Pre-Cooldown

| Item | Check | Notes |
|------|-------|-------|
| □ | Vacuum <10⁻⁵ torr | Leak check if not |
| □ | All instruments powered on | Verify communication |
| □ | Room temp tests pass | Coax continuity, DC biases |
| □ | Data logging started | Temperature, pressure |
| □ | Magnetic environment quiet | Turn off unnecessary equipment |
| □ | Emergency procedures reviewed | Know how to warm up safely |

### Cooldown

| Item | Check | Notes |
|------|-------|-------|
| □ | Monitor 1st stage temperature | Should reach 40-50K |
| □ | Monitor 2nd stage temperature | Should reach 4K |
| □ | Check for pressure rise | Indicates leak or virtual leak |
| □ | Verify thermometer readings | Consistent with expectations |
| □ | Check for vibration | Cryocooler mechanical noise |

### Post-Cooldown Verification

| Item | Check | Notes |
|------|-------|-------|
| □ | Base temperature reached | Typically <4.2K for good systems |
| □ | Temperature stable | <50 mK peak-to-peak |
| □ | DC biases verified | Currents reach DUT |
| □ | PCM structures functional | JJ I-V curves look right |
| □ | Noise floor acceptable | Check with no DUT signal |
| □ | Main circuit functional | Time for real testing! |

### Troubleshooting Quick Reference

| Symptom | Likely Cause | Check |
|---------|--------------|-------|
| **Won't cool below 10K** | Excessive heat load | Wiring, radiation, leak |
| **Temperature oscillating** | Cryocooler issue | Cold head, compressor |
| **High noise floor** | Ground loop | Star ground, shield |
| **No signal** | Open connection | Coax, wire bonds |
| **Wrong DC point** | Bias issue | Supplies, wiring R |
| **Intermittent failures** | Flux trapping | Shielding, cooldown field |
| **Margins too small** | Process or design | PCM data, simulation |

### Documentation

Always document:
1. **Wiring map**: Which connector pin goes where
2. **Cable lengths**: For timing-critical paths
3. **Thermal budget**: Calculated and measured loads
4. **Shielding configuration**: Materials, thickness, gaps
5. **Grounding scheme**: Where is star point, what floats
6. **Cooldown log**: Time, temperatures, issues

---
<a id="18-equipment-vendors"></a>
# 18. Equipment and Vendor Guide
---

Specific recommendations for building a cryogenic SCE test system.

### Cryocoolers

| Vendor | Model | Type | 4K Power | Notes |
|--------|-------|------|----------|-------|
| **Sumitomo** | RDK-408D2 | GM | 1.5 W | Workhorse, reliable |
| **Sumitomo** | RDK-415D | GM | 1.5 W | Newer version |
| **Cryomech** | PT410 | Pulse Tube | 1.0 W | Lower vibration |
| **Cryomech** | PT415 | Pulse Tube | 1.5 W | Higher capacity PT |
| **Sumitomo** | RP-082B2 | Pulse Tube | 0.9 W | Compact PT |
| **Bluefors** | SD series | Dilution | mK | For quantum work |

**Compressors**: Matched to cold head (Sumitomo CNA-11, Cryomech CP2870)

### Coaxial Cables

| Vendor | Part Number | Type | Z₀ | Notes |
|--------|-------------|------|-----|-------|
| **Micro-Coax** | UT-085-SS | Semi-rigid SS | 50Ω | Standard |
| **Micro-Coax** | UT-047-SS | Semi-rigid SS | 50Ω | Lower heat load |
| **Coax Co** | SC-086/50-SS-SS | Semi-rigid SS | 50Ω | Premium |
| **Coax Co** | SC-086/50-CN-CN | Semi-rigid CuNi | 50Ω | Moderate thermal |
| **Coax Co** | SC-086/50-NbTi-NbTi | Superconducting | 50Ω | Lowest loss below T_c |
| **Lake Shore** | Type C | Coax | 50Ω | Cryo-rated assemblies |
| **Keycom** | Various | Semi-rigid | 50Ω | Good NbTi options |

### DC Wiring

| Vendor | Type | AWG | Notes |
|--------|------|-----|-------|
| **Lake Shore** | Quad-Twist | 32-36 | Phosphor bronze, cryo-rated |
| **Lake Shore** | Cryogenic Wire | Various | Pre-spooled for cryo |
| **CMR Direct** | Manganin wire | 32-40 | Bulk spools |
| **Supercon Inc** | NbTi wire | 36+ | Superconducting |
| **California Fine Wire** | Custom | Various | Many alloys |

### Connectors

| Vendor | Series | Type | Notes |
|--------|--------|------|-------|
| **Amphenol** | SMA | RF coax | Standard, cheap |
| **Rosenberger** | SMA/2.92mm | RF coax | Higher quality |
| **Southwest Microwave** | 2.92mm/2.4mm | RF coax | Precision, expensive |
| **Samtec** | SFMC | Flex/board | Cryo-rated micro coax |
| **Hirose** | FH12/FH19 | Flex ZIF | Works at cryo (not rated) |
| **Omnetics** | Nano-D | High density DC | Mil-spec, cryo OK |
| **Glenair** | Micro-D | High density DC | Hermetic options |

### Filters

| Vendor | Type | Application | Notes |
|--------|------|-------------|-------|
| **Mini-Circuits** | LFCN/HFCN series | LP/HP coax | Cheap, work at cryo |
| **K&L Microwave** | Custom | LP/BP/HP | Higher performance |
| **Marki Microwave** | Reflectionless | LP | No reflections |
| **API Technologies** | Feedthrough caps | DC filtering | Pi-filters |
| **Spectrum Control** | EMI filters | DC feedthrough | Integrated filtering |
| **Tusonix** | Ceramic caps | Feedthrough | Simple, cheap |

### Attenuators

| Vendor | Model | Notes |
|--------|-------|-------|
| **XMA/Omni-Spectra** | Fixed SMA attenuators | Standard, work at cryo |
| **Weinschel** | Fixed/Variable | Higher precision |
| **Mini-Circuits** | VAT series | Variable |
| **Bluefors** | Cryo attenuators | Designed for dilution fridges |

### Amplifiers (Room Temperature)

| Vendor | Model | Bandwidth | Noise | Notes |
|--------|-------|-----------|-------|-------|
| **Mini-Circuits** | ZFL-500LN+ | DC-500 MHz | 2.9 dB NF | Low noise, cheap |
| **Mini-Circuits** | ZX60-P103LN+ | 0.5-10 GHz | 1.0 dB NF | Wideband |
| **RF Bay** | LNA series | Various | ~1 dB NF | Good value |
| **Pasternack** | PE15A series | Various | Various | Quick delivery |

### Amplifiers (Cryogenic)

| Vendor | Model | Temp | Notes |
|--------|-------|------|-------|
| **Low Noise Factory** | LNF-LNC | 4K | State-of-the-art HEMT |
| **Cosmic Microwave Tech** | CITLF series | 4K | HEMT, good availability |
| **Caltech** | Custom HEMT | 4K | Research-grade |
| **NRAO** | Custom | 4K | Radio astronomy heritage |

### Magnetic Shielding

| Vendor | Material | Notes |
|--------|----------|-------|
| **Magnetic Shield Corp** | Finemet tape/cans | **Preferred** - high µ, no anneal |
| **Vacuumschmelze** | Cryoperm 10 | Cryo-rated mu-metal |
| **Amuneal** | A4K | Optimized for 4K |
| **MuShield** | Mu-metal | Requires annealing |
| **Goodfellow** | Pb foil | Lead for SC shield |
| **Alfa Aesar** | Pb sheet | Lead sheet stock |

### Thermometry

| Vendor | Type | Range | Notes |
|--------|------|-------|-------|
| **Lake Shore** | DT-670 Si Diode | 1.4-500 K | Standard, cheap |
| **Lake Shore** | Cernox CX-1050 | 0.1-420 K | Low B-field sensitivity |
| **Lake Shore** | RX-202A RuO₂ | 0.05-40 K | Best for <4K |
| **Scientific Instruments** | Si diode | 1.4-500 K | Alternative vendor |
| **Lakeshore** | Model 336/372 | Controller | Temperature readout |

### Vacuum Equipment

| Vendor | Type | Notes |
|--------|------|-------|
| **Edwards** | nXDS dry scroll | Roughing pump |
| **Pfeiffer** | HiPace turbo | Turbo pump |
| **Agilent** | TwisTorr turbo | Magnetic bearing (low field) |
| **MKS/Granville-Phillips** | Ion gauge | Pressure measurement |
| **Kurt J. Lesker** | Feedthroughs | SMA, DC, optical |
| **MDC Vacuum** | Flanges, fittings | ISO/KF/CF components |
| **Swagelok** | VCR fittings | Gas handling |

### Thermal Management

| Vendor | Product | Notes |
|--------|---------|-------|
| **Apiezon** | N grease | Demountable thermal joints |
| **Apiezon** | H grease | Higher temp range |
| **Henkel** | Stycast 2850FT | Epoxy, permanent joints |
| **CMR Direct** | GE 7031 varnish | Wire anchoring |
| **Lakeshore** | Copper braid | Flexible thermal links |
| **TAI** | OFHC Cu | Custom machined parts |

### Test Instrumentation

| Vendor | Type | Notes |
|--------|------|-------|
| **Keysight** | BERT (M8020A) | Bit error rate testing |
| **Keysight** | AWG (M8190A) | Arbitrary waveform |
| **Keysight** | Scope (UXR) | High bandwidth scope |
| **Tektronix** | Scope | Alternative |
| **Stanford Research** | SR830 Lock-in | Low-level AC measurements |
| **Keithley** | 2182A | Nanovoltmeter (precision V measurement) |
| **Keithley** | 6221 | Precision current source |
| **Yokogawa** | GS200 | Precision DC source |
| **National Instruments** | DAQ | Data acquisition |



### Automated SCE Test Systems

Purpose-built systems for superconducting circuit characterization:

| System | Vendor | Features | Notes |
|--------|--------|----------|-------|
| **OCTOPUX** | RedHiTech | 32/64/128 channels, ±200µA-200mA bias, 0.5µV precision | **Industry standard** since 1997 |
| **ICE-T** | HYPRES | Cryogen-free, modular inserts, dual-insert option | Integrated cryostat + electronics |

#### OCTOPUX System

The [OCTOPUX](https://www.redhitech.com/octopux.html) is the de facto standard for automated superconducting circuit testing:

**Specifications:**
- **Channels**: 32, 64, or 128 universal I/O
- **DC bias**: 4 programmable ranges (±200 µA to ±200 mA), 16-bit resolution
- **Measurement**: Differential, 0.5 µV P-P accuracy in 4 kHz BW
- **DAQ**: Up to 2 MS/s, 10 µV P-P noise at 1 MS/s
- **Interference rejection**: Designed to minimize 50/60 Hz and RF pickup

**Key capabilities:**
- Automated I_c measurement for all junction types
- On-chip R and L extraction
- Multidimensional parametric margin analysis
- Real-time oscilloscope with spectral analysis
- MATLAB integration for custom scripts
- Self-calibration and diagnostics

**Best for**: RSFQ/ERSFQ testing, production characterization, margin mapping

**Note**: Limited for high-impedance loads (>1 kΩ) — not ideal for nanocryotron testing.

#### HYPRES ICE-T

The [ICE-T](https://www.hypres.com/products/integrated-cryogenic-electronics-test-bed-ice-t/) (Integrated Cryogenic Electronics Test-bed) combines cryostat and test electronics:

**Features:**
- Cryogen-free (closed-cycle cryocooler)
- Modular electrical inserts (standard or custom)
- Single insert (ISO-160) or dual insert (2× ISO-100) configurations
- Inserts can also work as LHe immersion probes

**Best for**: Labs wanting turnkey solution, rapid chip turnaround

#### DIY Alternative: FPGA + DAC/ADC

For custom or budget-constrained setups:

| Component | Options | Notes |
|-----------|---------|-------|
| **FPGA** | Xilinx RFSoC (ZCU111, ZCU216) | Integrated ADC/DAC |
| **DAC** | AD9164, DAC38RF | High-speed, multi-channel |
| **ADC** | AD9208, LTC2387 | High-speed sampling |
| **Bias DAC** | AD5791 (20-bit), DAC8565 | Precision DC |
| **Current source** | Custom or Keithley 6221 | Low-noise bias |

Building custom test systems requires significant software development but offers maximum flexibility for AQFP-specific requirements (e.g., 4-phase AC excitation control).

### Flex PCB Vendors

| Vendor | Notes |
|--------|-------|
| **Rigiflex** | US-based, good quality |
| **Epec** | US-based, fast turn |
| **PCBWay** | Overseas, cheap prototypes |
| **Sierra Circuits** | US, high-end |
| **Flexible Circuit Technologies** | Cryo experience |

### Other Useful Vendors

| Vendor | Products |
|--------|----------|
| **Lake Shore Cryotronics** | Full cryo catalog |
| **Janis Research** | Cryostats, probes |
| **Bluefors** | Dilution refrigerators |
| **ColdEdge Technologies** | Cryostat integration |
| **Precision Cryogenics** | Dewars, cryostats |
| **Cryofab** | Dewars, LHe storage |
| **Quantum Design** | PPMS, cryogenics |
| **ICE Oxford** | Cryostats |
| **Oxford Instruments** | Cryostats, magnets |
| **Thorlabs** | Optics, opto-mechanics |
| **Newport** | Optics, positioning |
| **McMaster-Carr** | Everything else |

---

### Recommended Starter Equipment List

For a basic AQFP test setup:

**Minimum viable system:**
| Item | Suggested | Qty |
|------|-----------|-----|
| Cryocooler | Sumitomo RDK-408D2 + CNA-11 | 1 |
| Vacuum chamber | Custom or vendor | 1 |
| Turbo pump | Pfeiffer HiPace 80 | 1 |
| Roughing pump | Edwards nXDS6i | 1 |
| Coax cables | UT-085-SS, 1m | 4-8 |
| DC wiring | Lake Shore Quad-Twist | 100m |
| Thermometers | DT-670 Si diode | 2-3 |
| Temp controller | Lake Shore 336 | 1 |
| Finemet tape | 25mm × 50m | 1 roll |
| Lead foil | 0.5mm × 300mm | 1 sheet |
| DC supplies | Yokogawa GS200 | 2-4 |
| Nanovoltmeter | Keithley 2182A | 1 |
| Scope | Keysight or Tek, ≥1 GHz | 1 |
| Pattern gen | Keysight or custom FPGA | 1 |

**Add for production testing:**
- BERT for BER measurement
- Multi-channel bias system
- Automated margin sweep software
- Additional cryocooler systems

---
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); color: white; padding: 30px; margin: 20px -10px -10px -10px; border-radius: 15px 15px 0 0; text-align: center;">

## Part 2 Summary

### Test Equipment Options
- **LHe dip probe**: Simple, cheap, fast cooldown — ideal for debug
- **Cryocooler system**: No consumables, automated — ideal for production

### Thermal Design Rules
- Intercept all wiring at 40K (1st stage has 30-50W available)
- Budget carefully at 4K (only 1-2W available)
- OFHC copper for thermal links, stainless for isolation
- Gold-plate all copper surfaces

### Signal Integrity
- Use appropriate coax (SS for low heat, NbTi for lowest loss)
- Match impedances (50Ω external, matched to PTL internal)
- Filter aggressively (DC lines: RC at each stage)

### Grounding
- Single star ground point (usually cryostat chassis)
- Shield cables at one end only
- Use differential signals where possible

### Magnetic Shielding
- Multi-layer: mu-metal (300K) + cryoperm (40K) + SC (4K)
- Cool through T_c in low field (<1 µT)
- Degauss mu-metal after handling

### Key Numbers to Remember
| Parameter | Target |
|-----------|--------|
| Base temperature | <4.2 K |
| 4K heat budget | <50% of rated |
| Vacuum | <10⁻⁵ torr |
| Field at chip during T_c | <1 µT |
| Connector torque (SMA) | 8 in-lb |
| Contraction allowance | 1-2% |

</div>